In [ ]:
import os

import requests
from dotenv import load_dotenv
import time
import csv



load_dotenv(".env")
api_key_TMDB = os.getenv("TMDB_API_KEY")


if not api_key_TMDB:
    print("API KEY not found check env file")
    exit()
else:
    print("TMDB KEY loaded")



#OMDb key
load_dotenv(".env")
api_key_OMDB = os.getenv("OMDB_API_KEY")


if not api_key_OMDB:
    print("API KEY not found check env file")
    exit()
else:
        print("OMDB KEY loaded")

TMDB KEY loaded
OMDB KEY loaded


In [ ]:
def load_movies_tmdb_for_research(api_key, start_year=2006, end_year=2026, min_page=1, max_page=5):
    """
    Loads top movies per year 
    Returns a standard Python list containing dictionaries for each movie.
    """
    all_top_movies = []
    base_url = "https://api.themoviedb.org/3"
    discover_url = f"{base_url}/discover/movie"

    print(f"Start download for the years {start_year} to {end_year}")

    # Loop through each year
    for year in range(start_year, end_year + 1):
        print(f"Loading data for year: {year}")
        
        start_date = f"{year}-01-01"
        end_date = f"{year}-12-31"

        for actual_page in range(min_page, max_page + 1):
            
            # filter
            parameter = {
                "api_key": api_key,
                "primary_release_date.gte": start_date,
                "primary_release_date.lte": end_date,
                "language": "en-US", 
                "sort_by": "revenue.desc", 
                "page": actual_page,
                "vote_average.gte": 0.1,
                "vote_count.gte":5 
            }

            
            response = requests.get(discover_url, params=parameter)

            if response.status_code == 200:
                data = response.json()
                
                if actual_page == min_page:
                    print(f"Total movies found for {year}: {data.get('total_results')} on total pages: {data.get('total_pages')}")
                
                movies_on_page = data.get("results", [])
                
                if not movies_on_page:
                    break

                # For each film per page
                for movie in movies_on_page:
                    tmdb_id = movie.get("id")
                    imdb_id = "Not found"
                    revenue = 0
                    budget = 0
                    genres = [] 
                    
                    
                    detail_url = f"{base_url}/movie/{tmdb_id}"
                    detail_params = {"api_key": api_key}
                    
                    try:
                        detail_response = requests.get(detail_url, params=detail_params)
                        if detail_response.status_code == 200:
                            detail_data = detail_response.json()
                            imdb_id = detail_data.get("imdb_id", "Not found")
                            revenue = detail_data.get("revenue", 0)
                            budget = detail_data.get("budget", 0)
                            genres = [genre['name'] for genre in detail_data.get("genres", [])]
                    except Exception:
                        pass
                        
                    
                    all_top_movies.append({
                        "release_year": year, 
                        "tmdb_id": tmdb_id,
                        "imdb_id": imdb_id,
                        "title": movie.get("title"),
                        "genres": genres, 
                        "budget": budget,
                        "box_office_revenue": revenue,
                        "vote_average": movie.get("vote_average"),
                        "vote_count": movie.get("vote_count")
                    })
                    
                    # Spam protection for detail requests
                    time.sleep(0.05)
                    
                print(f"Loaded page {actual_page} for {year}.")
            
            else:
                print(f"Error on page {actual_page}: Code {response.status_code}")
                break
                
            # Spam protection between full pages
            time.sleep(0.3)
            
    print(f"Finished! Downloaded {len(all_top_movies)} movies total.")
    
    
    return all_top_movies

In [ ]:


def save_movies_to_csv(movie_list, filename="research_movies_data.csv"):
    """
    Saves movie list into csv allowing caching of result
   
    """
    print(f"Save data to {filename}")
    
    
    file_exists = os.path.isfile(filename)
    
   
    with open(filename, mode='a', newline='', encoding='utf-8') as file:
        
       
        columns = [
            'tmdb_id', 'imdb_id', 'title', 'release_year', 'genres', 'budget', 'box_office_revenue', 'vote_average', 'vote_count']
        
        writer = csv.DictWriter(file, fieldnames=columns)
        
        
        if not file_exists:
            writer.writeheader()
            
        
        for movie in movie_list:
            
            # Convert genre list to a comma-separated string for the CSV
            genre_data = movie.get('genres', [])
            if isinstance(genre_data, list) and len(genre_data) > 0:
                genre_string = ", ".join(genre_data)
            else:
                genre_string = "Not found"
                
            
            writer.writerow({
                'tmdb_id': movie.get('tmdb_id', 'N/A'),
                'imdb_id': movie.get('imdb_id', 'Not found'),
                'title': movie.get('title', 'N/A'),
                'release_year': movie.get('release_year', 'N/A'),
                'genres': genre_string,
                'budget': movie.get('budget', 0),
                'box_office_revenue': movie.get('revenue', 0),
                'vote_average': movie.get('vote_average', 0.0),
                'vote_count': movie.get('vote_count', 0)
            })
            
    print("Finished")

In [ ]:
#testblock

movies_2023 = load_movies_tmdb_for_research(api_key=api_key_TMDB,start_year=2025,end_year=2025,min_page=1,max_page=125
)


save_movies_to_csv(movies_2023, filename="movies_only_2025.csv")